# SGGF-Net Training Notebook (Google Colab - T4 GPU)

**3-Stage Training Strategy optimized for Colab T4 GPU**

- Stage 1: Baseline Faster-RCNN (8 epochs, ~15-20 min)
- Stage 2: Enable GFEM (6 epochs, ~12-15 min)
- Stage 3: Enable NDPA + ARPM (4 epochs, ~8-10 min)

**Total time: ~35-45 minutes on T4 GPU**

**Setup:**
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells

In [ ]:
# Step 1: Mount Drive and clone repository
import os
from google.colab import drive

print("=" * 70)
print("SETUP")
print("=" * 70)

# Mount Drive
try:
    if os.path.exists('/content/drive/MyDrive'):
        print('✓ Google Drive already mounted')
    else:
        drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted')
except Exception as e:
    print(f'⚠ Drive mounting failed: {e}')

# Clone repository
if not os.path.exists('SGGF-Net'):
    print('\n📦 Cloning repository...')
    !git clone https://github.com/HarishSankarK/SGGF-Net.git
    os.chdir('SGGF-Net')
else:
    os.chdir('SGGF-Net')

print(f'\n✓ Working directory: {os.getcwd()}')
print("=" * 70)

In [ ]:
# Step 2: Install dependencies and verify GPU
print("=" * 70)
print("INSTALLING DEPENDENCIES")
print("=" * 70)

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install numpy pillow opencv-python tqdm matplotlib scipy --quiet

import torch
print(f"\n✓ PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("⚠ No GPU! Enable GPU in Runtime settings")
print("=" * 70)

## Stage 1: Baseline Faster-RCNN

In [ ]:
# Stage 1 Training
import subprocess
import os

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print("=" * 70)
print("STAGE 1: BASELINE FASTER-RCNN")
print("=" * 70 + "\n")

cmd = ['python', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '1', '--subset_ratio', '0.35']
subprocess.run(cmd, check=True)

## Stage 2: Enable GFEM

In [ ]:
# Stage 2 Training
import subprocess
import os

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
stage1_best = os.path.join(checkpoint_dir, 'stage1_best.pth')

if not os.path.exists(stage1_best):
    print(f"⚠ {stage1_best} not found. Run Stage 1 first!")
else:
    print("=" * 70)
    print("STAGE 2: ENABLE GFEM")
    print("=" * 70 + "\n")
    cmd = ['python', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '2', '--resume', stage1_best, '--subset_ratio', '0.35']
    subprocess.run(cmd, check=True)

## Stage 3: Enable NDPA + ARPM

In [ ]:
# Stage 3 Training
import subprocess
import os

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
stage2_best = os.path.join(checkpoint_dir, 'stage2_best.pth')

if not os.path.exists(stage2_best):
    print(f"⚠ {stage2_best} not found. Run Stage 2 first!")
else:
    print("=" * 70)
    print("STAGE 3: ENABLE NDPA + ARPM")
    print("=" * 70 + "\n")
    cmd = ['python', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '3', '--resume', stage2_best, '--subset_ratio', '0.35']
    subprocess.run(cmd, check=True)

## Evaluate Final Model

In [ ]:
# Evaluation
import subprocess
import os
import torch

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
final_checkpoint = os.path.join(checkpoint_dir, 'stage3_best.pth')
if not os.path.exists(final_checkpoint):
    final_checkpoint = os.path.join(checkpoint_dir, 'stage3_latest.pth')

if not os.path.exists(final_checkpoint):
    print(f"⚠ {final_checkpoint} not found. Complete all stages first!")
else:
    device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 70)
    print("EVALUATION")
    print("=" * 70 + "\n")
    cmd = ['python', 'scripts/evaluate.py', '--dataset', 'hituav', '--data_dir', 'data/hit-uav', '--checkpoint', final_checkpoint, '--num_classes', '6', '--batch_size', '1', '--max_size', '640', '--split', 'test', '--device', device_str]
    subprocess.run(cmd, check=True)